In [1]:
# =============================================================
# 🧠 BRAIN TUMOR MRI COMPLETE PIPELINE
# Classification + Training + Testing + GradCAM + LIME + Superpixel
# =============================================================

import os, glob, warnings, random, cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_pre
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_pre

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D,
    BatchNormalization, Activation, Input,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import regularizers
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# XAI imports
import lime
import lime.lime_image as lime_lib
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage.segmentation import quickshift, mark_boundaries

warnings.filterwarnings("ignore")

# =============================================================
# CONFIG
# =============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["glioma","meningioma","notumor","pituitary"]
NUM_CLASSES = 4
BATCH_SIZE = 32

OUTPUT_DIR="/kaggle/working/outputs"
TRAIN_DIR="/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training"
TEST_DIR="/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing"
os.makedirs(OUTPUT_DIR,exist_ok=True)

MODEL_CONFIG = {
    "vgg16": {
        "model_fn": VGG16,
        "preprocess_fn": vgg16_pre,
        "img_size": (224,224),
        "unfreeze_p2": "block5_conv1",
        "unfreeze_p3": "block4_conv1",
        "phase1_lr": 5e-4,
        "phase2_lr": 5e-5,
        "phase3_lr": 1e-5,
        "phase1_epochs": 30,
        "phase2_epochs": 40,
        "phase3_epochs": 30,
        "unfreeze_type": "named"
    },
    "resnet50": {
        "model_fn": ResNet50,
        "preprocess_fn": resnet50_pre,
        "img_size": (224,224),
        "unfreeze_p2": "conv5_block1_1_conv",
        "unfreeze_p3": "conv4_block1_1_conv",
        "phase1_lr": 5e-4,
        "phase2_lr": 5e-5,
        "phase3_lr": 1e-5,
        "phase1_epochs": 30,
        "phase2_epochs": 50,
        "phase3_epochs": 35,
        "unfreeze_type": "named"
    },
    "efficientnetb0": {
        "model_fn": EfficientNetB0,
        "preprocess_fn": effnet_pre,
        "img_size": (240,240),
        "unfreeze_p2": 80,
        "unfreeze_p3": 150,
        "phase1_lr": 5e-4,
        "phase2_lr": 5e-5,
        "phase3_lr": 1e-5,
        "phase1_epochs": 30,
        "phase2_epochs": 50,
        "phase3_epochs": 35,
        "unfreeze_type": "count"
    }
}
print("✅ Config ready")
print(f"   GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")

# =============================================================
# HELPERS
# =============================================================
def preprocess_for_model(img, model_name):
    fn = MODEL_CONFIG[model_name]["preprocess_fn"]
    return fn(img)

def get_target_size(model_name):
    return MODEL_CONFIG[model_name]["img_size"]

def get_image_files(cls_dir, n=2):
    extensions = ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(cls_dir, ext)))
    files = sorted(files)
    if len(files)==0: return []
    if len(files)<=n: return files
    rng = random.Random(SEED)
    return rng.sample(files, n)

def load_as_uint8(img_path, model_name):
    sz = get_target_size(model_name)
    img = load_img(img_path, target_size=sz)
    arr = img_to_array(img)
    return arr.astype(np.uint8)

def load_for_predict(img_path, model_name):
    sz = get_target_size(model_name)
    img = load_img(img_path, target_size=sz)
    arr = img_to_array(img)
    arr = preprocess_for_model(arr, model_name)
    return np.expand_dims(arr,0).astype(np.float32)

def load_rgb_display(img_path, model_name):
    sz = get_target_size(model_name)
    bgr = cv2.imread(img_path)
    if bgr is None:
        img = load_img(img_path, target_size=sz)
        return img_to_array(img).astype(np.uint8)
    return cv2.cvtColor(cv2.resize(bgr, sz), cv2.COLOR_BGR2RGB)

def get_prediction(model, img_path, model_name):
    sz = get_target_size(model_name)
    img = load_img(img_path, target_size=sz)
    arr = img_to_array(img)
    processed = preprocess_for_model(arr, model_name)
    batch = np.expand_dims(processed, axis=0)
    probs = model.predict(batch, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_conf  = float(probs[pred_idx]) * 100.0
    pred_label = CLASS_NAMES[pred_idx]
    return pred_idx, pred_conf, pred_label

# =============================================================
# DATA GENERATORS + CLASS WEIGHTS
# =============================================================
def get_generators(model_name):
    cfg = MODEL_CONFIG[model_name]
    img_size = cfg["img_size"]
    pre_fn = cfg["preprocess_fn"]
    train_datagen = ImageDataGenerator(
        preprocessing_function=pre_fn,
        rotation_range=15,
        width_shift_range=0.10,
        height_shift_range=0.10,
        shear_range=0.1,
        zoom_range=0.10,
        horizontal_flip=True,
        vertical_flip=False,
        fill_mode="nearest",
        validation_split=0.2
    )
    test_datagen = ImageDataGenerator(preprocessing_function=pre_fn)
    common = dict(target_size=img_size, batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES, seed=SEED)
    train_gen = train_datagen.flow_from_directory(TRAIN_DIR, subset="training", shuffle=True, **common)
    val_gen   = train_datagen.flow_from_directory(TRAIN_DIR, subset="validation", shuffle=False, **common)
    test_gen  = test_datagen.flow_from_directory(TEST_DIR, shuffle=False, **common)
    return train_gen, val_gen, test_gen

def compute_class_weights(train_dir):
    counts = {}
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(train_dir, cls)
        counts[cls] = len([f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    total = sum(counts.values())
    n_cls = len(CLASS_NAMES)
    weights = {i: total / (n_cls * counts[cls]) for i, cls in enumerate(CLASS_NAMES)}
    weights[0] = weights[0] * 1.3  # boost glioma
    print("\n Class weights (glioma boosted):")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"   {cls:12s}: {weights[i]:.4f} ({counts[cls]} images)")
    return weights

# =============================================================
# MODEL / UNFREEZE / CALLBACKS
# =============================================================
def build_model(model_name):
    cfg = MODEL_CONFIG[model_name]
    input_shape = (*cfg["img_size"], 3)
    base = cfg["model_fn"](weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    inputs = Input(shape=input_shape)
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D(name='gap')(x)
    x = Dense(256, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(0.4)(x)
    outputs = Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    model = Model(inputs, outputs)
    return model, base

def unfreeze_layers(base, model_name, phase):
    cfg = MODEL_CONFIG[model_name]
    if cfg["unfreeze_type"] == "named":
        flag = False
        for layer in base.layers:
            if layer.name == cfg[f"unfreeze_p{phase}"]:
                flag = True
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
            else:
                layer.trainable = flag
    else:
        total = len(base.layers)
        for i, layer in enumerate(base.layers):
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
            else:
                layer.trainable = (i >= total - cfg[f"unfreeze_p{phase}"])

def get_callbacks(name, phase):
    ckpt = os.path.join(OUTPUT_DIR, f"{name}_p{phase}.keras")
    return [
        ModelCheckpoint(ckpt, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7)
    ]

# =============================================================
# TRAINING & TESTING & TTA
# =============================================================
def predict_with_tta(model, test_gen, model_name, n_augments=10):
    cfg = MODEL_CONFIG[model_name]
    pre_fn = cfg["preprocess_fn"]
    img_size = cfg["img_size"]
    tta_kwargs = dict(
        rotation_range=10, width_shift_range=0.05, height_shift_range=0.05, zoom_range=0.05,
        horizontal_flip=True, fill_mode="nearest", preprocessing_function=pre_fn
    )
    tta_datagen   = ImageDataGenerator(**tta_kwargs)
    clean_datagen = ImageDataGenerator(preprocessing_function=pre_fn)
    common = dict(target_size=img_size, batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES, shuffle=False)
    clean_gen = clean_datagen.flow_from_directory(TEST_DIR, **common)
    steps = int(np.ceil(clean_gen.samples / BATCH_SIZE))
    all_preds = model.predict(clean_gen, steps=steps, verbose=0)
    for i in range(n_augments):
        aug_gen = tta_datagen.flow_from_directory(TEST_DIR, seed=SEED + i + 1, **common)
        preds = model.predict(aug_gen, steps=steps, verbose=0)
        all_preds += preds
    all_preds /= (n_augments + 1)
    y_pred = np.argmax(all_preds, axis=1)
    y_true = clean_gen.classes[:len(y_pred)]
    return y_true, y_pred

def plot_curves(model_name, *histories):
    acc = []; val_acc = []; loss = []; val_loss = []; splits = []
    for h in histories:
        if h is None: continue
        splits.append(len(acc))
        acc.extend(h.history["accuracy"])
        val_acc.extend(h.history["val_accuracy"])
        loss.extend(h.history["loss"])
        val_loss.extend(h.history["val_loss"])
    epochs = range(1, len(acc)+1)
    clrs = ["green", "blue", "orange"]
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, tr, vl, title in [(axes[0], acc, val_acc, "Accuracy"), (axes[1], loss, val_loss, "Loss")]:
        ax.plot(epochs, tr, 'b-', label="Train", linewidth=2)
        ax.plot(epochs, vl, 'r-', label="Val", linewidth=2)
        for i, sp in enumerate(splits[1:], 1):
            ax.axvline(sp+1, color=clrs[i % len(clrs)], linestyle="--", linewidth=1.5, label=f"Phase {i+1}")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"{model_name}_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def evaluate(model, model_name, test_gen, use_tta=True):
    print(f"\n   Evaluating {'with TTA' if use_tta else 'standard'}...")
    if use_tta:
        y_true, y_pred = predict_with_tta(model, test_gen, model_name, n_augments=10)
    else:
        test_gen.reset()
        steps = int(np.ceil(test_gen.samples / BATCH_SIZE))
        probs = model.predict(test_gen, steps=steps, verbose=1)
        y_pred = np.argmax(probs, axis=1)
        y_true = test_gen.classes[:len(y_pred)]
    overall = accuracy_score(y_true, y_pred)
    print(f"   Test Accuracy : {overall*100:.2f}%")
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linewidths=0.5, annot_kws={"size": 12})
    ax.set_title(f"{model_name.upper()} Confusion Matrix\nAccuracy: {overall*100:.2f}%", fontsize=12, fontweight="bold")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    cm_path = os.path.join(OUTPUT_DIR, f"{model_name}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Confusion matrix saved: {cm_path}")
    return overall, y_true, y_pred

def train_and_evaluate(model_name):
    cfg = MODEL_CONFIG[model_name]
    print(f"\n{'='*55}")
    print(f"   {model_name.upper()}")
    print(f"{'='*55}")
    train_gen, val_gen, test_gen = get_generators(model_name)
    class_weights = compute_class_weights(TRAIN_DIR)
    model, base = build_model(model_name)
    
    print(f"\n   Phase 1 — head only | lr={cfg['phase1_lr']}")
    model.compile(optimizer=Adam(learning_rate=cfg["phase1_lr"]), loss="categorical_crossentropy", metrics=["accuracy"])
    h1 = model.fit(train_gen, epochs=cfg["phase1_epochs"], validation_data=val_gen, class_weight=class_weights, callbacks=get_callbacks(model_name, 1), verbose=1)
    p1_best = max(h1.history["val_accuracy"])
    print(f"    Phase 1 best: {p1_best*100:.2f}%")
    
    print(f"\n   Phase 2 — fine-tune | lr={cfg['phase2_lr']}")
    unfreeze_layers(base, model_name, phase=2)
    model.compile(optimizer=Adam(learning_rate=cfg["phase2_lr"]), loss="categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(train_gen, epochs=cfg["phase2_epochs"], validation_data=val_gen, class_weight=class_weights, callbacks=get_callbacks(model_name, 2), verbose=1)
    p2_best = max(h2.history["val_accuracy"])
    print(f"    Phase 2 best: {p2_best*100:.2f}%")
    
    h3 = None; p3_best = None
    run_phase3 = (model_name != "vgg16") or (p2_best < 0.96)
    if run_phase3:
        print(f"\n   Phase 3 — deeper | lr={cfg['phase3_lr']}")
        unfreeze_layers(base, model_name, phase=3)
        model.compile(optimizer=Adam(learning_rate=cfg["phase3_lr"]), loss="categorical_crossentropy", metrics=["accuracy"])
        h3 = model.fit(train_gen, epochs=cfg["phase3_epochs"], validation_data=val_gen, class_weight=class_weights, callbacks=get_callbacks(model_name, 3), verbose=1)
        p3_best = max(h3.history["val_accuracy"])
        print(f"    Phase 3 best: {p3_best*100:.2f}%")
    else:
        print(f"\n     Phase 3 skipped (val={p2_best*100:.2f}% ≥ 96%)")
        
    path = os.path.join(OUTPUT_DIR, f"{model_name}_v2_final.keras")
    model.save(path)
    print(f"   Model saved: {path}")
    plot_curves(model_name, h1, h2, h3)
    
    overall_std, _, _ = evaluate(model, model_name, test_gen, use_tta=False)
    overall_tta, _, _ = evaluate(model, model_name, test_gen, use_tta=True)
    overall = max(overall_std, overall_tta)
    return model, overall

# =============================================================
# GRAD-CAM
# =============================================================
def find_base_model(model):
    for layer in model.layers:
        if hasattr(layer, "layers") and len(layer.layers) > 5:
            return layer
    raise ValueError(f"Base model not found in {model.name}.")

def find_gradcam_target_layer(base_model, model_name):
    layer_names = [l.name for l in base_model.layers]
    if model_name == "resnet50":
        for name in ["conv5_block3_add", "add_15", "conv5_block3_out"]:
            if name in layer_names: return name
    elif model_name == "efficientnetb0":
        if "top_conv" in layer_names: return "top_conv"
    elif model_name == "vgg16":
        if "block5_conv3" in layer_names: return "block5_conv3"
    for layer in reversed(base_model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D, tf.keras.layers.Add)):
            return layer.name
    raise ValueError(f"No target layer for {model_name}.")

def compute_gradcam(model, base_model, target_layer_name, img_array, pred_index=None):
    try:
        target_layer = base_model.get_layer(target_layer_name)
        inner_model = tf.keras.Model(
            inputs=base_model.inputs,
            outputs=[target_layer.output, base_model.output]
        )
        head_layers = []
        for layer in model.layers:
            if not isinstance(layer, tf.keras.layers.InputLayer) and layer.name != base_model.name:
                head_layers.append(layer)
        
        # Explicit float32 cast for safe gradients
        img_tensor = tf.cast(img_array, tf.float32)
        
        with tf.GradientTape() as tape:
            conv_outputs, x = inner_model(img_tensor, training=False)
            tape.watch(conv_outputs)
            for layer in head_layers:
                x = layer(x, training=False)
            predictions = x
            if pred_index is None:
                pred_index = int(tf.argmax(predictions[0]).numpy())
            class_score = predictions[:, pred_index]
            
        grads = tape.gradient(class_score, conv_outputs)
        if grads is not None:
            weights = tf.reduce_mean(grads, axis=(0, 1, 2))
            cam = tf.reduce_sum(tf.multiply(weights, conv_outputs[0]), axis=-1)
            cam = tf.nn.relu(cam)
            cam_np = cam.numpy()
            mn, mx = cam_np.min(), cam_np.max()
            if (mx - mn) > 1e-8:
                cam_np = (cam_np - mn) / (mx - mn)
            else:
                cam_np = np.zeros_like(cam_np)
            pred_conf = float(predictions[0][pred_index].numpy()) * 100
            return cam_np, int(pred_index), pred_conf, True
    except Exception as e:
        print(f"     Grad-CAM extraction failed: {e}")
        
    try:
        predictions = model.predict(img_array, verbose=0)
        if pred_index is None:
            pred_index = int(np.argmax(predictions[0]))
        pred_conf = float(predictions[0][pred_index]) * 100
        return np.zeros((7, 7)), int(pred_index), pred_conf, False
    except Exception:
        return np.zeros((7, 7)), 0, 0.0, False

def build_overlay(orig_rgb, heatmap, alpha=0.5):
    H, W = orig_rgb.shape[:2]
    hm_up    = cv2.resize(heatmap.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    hm_up    = np.clip(hm_up, 0, 1)
    hm_uint8 = np.uint8(255 * hm_up)
    hm_jet   = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)
    hm_jet   = cv2.cvtColor(hm_jet, cv2.COLOR_BGR2RGB)
    overlay  = np.uint8(hm_jet * alpha + orig_rgb * (1 - alpha))
    return overlay, hm_up

def run_gradcam_model(model, model_name, test_dir=TEST_DIR, output_dir=OUTPUT_DIR, samples_per_class=2):
    print(f"\n{'─'*55}")
    print(f"   Grad-CAM: {model_name.upper()}")
    try:
        base_model = find_base_model(model)
        target_layer_name = find_gradcam_target_layer(base_model, model_name)
    except Exception as e:
        print(f"    Setup failed: {e}")
        return
        
    n_rows = len(CLASS_NAMES)
    n_cols = samples_per_class * 3
    fig    = plt.figure(figsize=(5 * n_cols, 5 * n_rows))
    gs     = gridspec.GridSpec(n_rows, n_cols, wspace=0.05, hspace=0.4)
    for row, cls_name in enumerate(CLASS_NAMES):
        cls_dir   = os.path.join(test_dir, cls_name)
        img_files = get_image_files(cls_dir, samples_per_class)
        if not img_files: continue
        for s, img_path in enumerate(img_files):
            col = s * 3
            try:
                img_arr  = load_for_predict(img_path, model_name)
                orig_rgb = load_rgb_display(img_path, model_name)
                heatmap, pred_idx, pred_conf, got_grads = compute_gradcam(model, base_model, target_layer_name, img_arr)
                overlay, hm_up = build_overlay(orig_rgb, heatmap)
                pred_label = CLASS_NAMES[pred_idx]
                correct    = (pred_label == cls_name)
                t_color    = "green" if correct else "red"
                
                ax0 = fig.add_subplot(gs[row, col])
                ax0.imshow(orig_rgb)
                ax0.set_title(f"True: {cls_name}\nPred: {pred_label} ({pred_conf:.1f}%)", fontsize=9, color=t_color, fontweight="bold")
                ax0.axis("off")
                
                method = "Grad-CAM" if got_grads else "Activation"
                ax1 = fig.add_subplot(gs[row, col + 1])
                im1 = ax1.imshow(hm_up, cmap="jet", vmin=0, vmax=1)
                ax1.set_title(f"{method} Heatmap", fontsize=9)
                ax1.axis("off")
                plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
                
                ax2 = fig.add_subplot(gs[row, col + 2])
                ax2.imshow(overlay)
                ax2.set_title(f"{method} on MRI", fontsize=9)
                ax2.axis("off")
            except Exception:
                pass
                
    fig.suptitle(f"{model_name.upper()} — Grad-CAM\nLayer: {target_layer_name}", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    path = os.path.join(output_dir, f"{model_name}_gradcam.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def run_gradcam_all(models_dict, test_dir=TEST_DIR, output_dir=OUTPUT_DIR, samples_per_class=2):
    for name, model in models_dict.items():
        run_gradcam_model(model, name, test_dir, output_dir, samples_per_class)

# =============================================================
# LIME & SUPERPIXEL WEIGHTS
# =============================================================
def make_lime_predict_fn(model, model_name):
    pre_fn = MODEL_CONFIG[model_name]["preprocess_fn"]
    def predict_fn(images_uint8):
        imgs = np.array(images_uint8, dtype=np.float32)
        if pre_fn is not None:
            processed = np.stack([pre_fn(img.copy()) for img in imgs], axis=0)
        else:
            processed = imgs / 255.0
        return model.predict(processed, verbose=0)
    return predict_fn

def run_lime_model(model, model_name, test_dir=TEST_DIR, output_dir=OUTPUT_DIR, samples_per_class=2, num_samples=1000, num_features=5):
    print(f"\n{'─'*55}")
    print(f"   LIME: {model_name.upper()}")
    predict_fn = make_lime_predict_fn(model, model_name)
    segmenter = SegmentationAlgorithm('quickshift', kernel_size=4, max_dist=200, ratio=0.2)
    explainer = lime_lib.LimeImageExplainer(random_state=SEED)
    n_rows = len(CLASS_NAMES)
    n_cols = samples_per_class * 5
    fig    = plt.figure(figsize=(4 * n_cols, 5 * n_rows))
    gs     = gridspec.GridSpec(n_rows, n_cols, wspace=0.1, hspace=0.4)
    for row, cls_name in enumerate(CLASS_NAMES):
        cls_dir   = os.path.join(test_dir, cls_name)
        img_files = get_image_files(cls_dir, samples_per_class)
        if not img_files: continue
        for s, img_path in enumerate(img_files):
            col = s * 5
            try:
                img_uint8 = load_as_uint8(img_path, model_name)
                pred_idx, pred_conf, pred_label = get_prediction(model, img_path, model_name)
                t_color = "green" if pred_label == cls_name else "red"
                explanation = explainer.explain_instance(
                    img_uint8, predict_fn, top_labels=NUM_CLASSES, hide_color=0,
                    num_samples=num_samples, segmentation_fn=segmenter, random_seed=SEED)
                    
                ax0 = fig.add_subplot(gs[row, col])
                ax0.imshow(img_uint8)
                ax0.set_title(f"True: {cls_name}\nPred: {pred_label} ({pred_conf:.1f}%)", fontsize=8, color=t_color)
                ax0.axis("off")
                
                ax1 = fig.add_subplot(gs[row, col+1])
                segs = quickshift(img_uint8, kernel_size=4, max_dist=200, ratio=0.2)
                n_segs = len(np.unique(segs))
                ax1.imshow(mark_boundaries(img_uint8.astype(np.float32)/255.0, segs, color=(1,1,0), mode="thick"))
                ax1.set_title(f"Quickshift\n({n_segs} segments)", fontsize=8)
                ax1.axis("off")
                
                ax2 = fig.add_subplot(gs[row, col+2])
                temp_pos, mask_pos = explanation.get_image_and_mask(
                        label=pred_idx, positive_only=True, num_features=num_features, hide_rest=False)
                disp = temp_pos.astype(np.float32)
                disp = disp / 255.0 if disp.max()>1.0 else disp
                disp = np.clip(disp,0,1)
                ax2.imshow(mark_boundaries(disp, mask_pos, color=(0,1,0), mode="thick"))
                ax2.set_title(f"Positive\n(top {num_features})", fontsize=8)
                ax2.axis("off")
                
                ax3 = fig.add_subplot(gs[row, col+3])
                temp_pn, mask_pn = explanation.get_image_and_mask(
                        label=pred_idx, positive_only=False, num_features=num_features, hide_rest=False)
                _, mask_pos_only = explanation.get_image_and_mask(
                        label=pred_idx, positive_only=True, num_features=num_features, hide_rest=False)
                neg_mask = np.logical_and(mask_pn.astype(bool), ~mask_pos_only.astype(bool)).astype(int)
                disp_pn = temp_pn.astype(np.float32)
                disp_pn = disp_pn/255.0 if disp_pn.max()>1.0 else disp_pn
                disp_pn = np.clip(disp_pn,0,1)
                disp_pn = mark_boundaries(disp_pn, mask_pos_only, color=(0,1,0), mode="thick")
                disp_pn = mark_boundaries(disp_pn, neg_mask, color=(1,0,0), mode="thick")
                ax3.imshow(disp_pn)
                ax3.set_title("Pos(green)\n+Neg(red)", fontsize=8)
                ax3.axis("off")
                
                ax4 = fig.add_subplot(gs[row, col+4])
                w_dict = dict(explanation.local_exp[pred_idx])
                lime_segs  = explanation.segments
                weight_map = np.zeros_like(lime_segs, dtype=np.float64)
                for seg_id, weight in w_dict.items():
                    weight_map[lime_segs == seg_id] = weight
                vmax = float(np.max(np.abs(weight_map)))
                vmax = vmax if vmax > 1e-8 else 1.0
                im = ax4.imshow(weight_map, cmap="RdYlGn", vmin=-vmax, vmax=vmax)
                ax4.axis("off")
                plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)
            except Exception:
                pass
                
    fig.suptitle(f"{model_name.upper()} — LIME Explanations\nQuickshift | Green=supports | Red=contradicts", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    save_path = os.path.join(output_dir, f"{model_name}_lime.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   LIME saved: {save_path}")

def run_lime_all(models_dict, test_dir=TEST_DIR, output_dir=OUTPUT_DIR, samples_per_class=2, num_samples=1000, num_features=5):
    for name, model in models_dict.items():
        run_lime_model(model, name, test_dir, output_dir, samples_per_class, num_samples, num_features)

def plot_superpixel_weights(model, model_name, img_path, output_dir=OUTPUT_DIR, num_samples=1000, top_n=15):
    predict_fn = make_lime_predict_fn(model, model_name)
    segmenter  = SegmentationAlgorithm('quickshift', kernel_size=4, max_dist=200, ratio=0.2)
    explainer  = lime_lib.LimeImageExplainer(random_state=SEED)
    
    img_uint8 = load_as_uint8(img_path, model_name)
    pred_idx, pred_conf, pred_label = get_prediction(model, img_path, model_name)
    
    explanation = explainer.explain_instance(
        img_uint8, predict_fn, top_labels=NUM_CLASSES, hide_color=0, num_samples=num_samples,
        segmentation_fn=segmenter, random_seed=SEED)
        
    local_weights = explanation.local_exp.get(pred_idx, [])
    
    # FATAL FLAW FIXED: Empty array check added
    if not local_weights:
        print(f"    No LIME weights found for pred_idx={pred_idx}. Skipping plotting.")
        return
        
    weights = sorted(local_weights, key=lambda x: abs(x[1]), reverse=True)[:top_n]
    seg_ids = [f"S{w[0]}" for w in weights]
    values  = [w[1] for w in weights]
    colours = ["#2ca02c" if v > 0 else "#d62728" for v in values]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(seg_ids[::-1], values[::-1], color=colours[::-1], edgecolor="white", height=0.6)
    axes[0].axvline(0, color="black", linewidth=0.8)
    axes[0].set_xlabel("Weight (w)", fontsize=11)
    axes[0].set_ylabel("Superpixel ID", fontsize=11)
    axes[0].set_title(f"Top {top_n} Weights", fontsize=11)
    axes[0].grid(axis="x", alpha=0.3)
    
    lime_segs  = explanation.segments
    w_dict     = dict(explanation.local_exp[pred_idx])
    weight_map = np.zeros_like(lime_segs, dtype=np.float64)
    for seg_id, weight in w_dict.items():
        weight_map[lime_segs == seg_id] = weight
        
    vmax = float(np.max(np.abs(weight_map)))
    vmax = vmax if vmax > 1e-8 else 1.0
    im = axes[1].imshow(weight_map, cmap="RdYlGn", vmin=-vmax, vmax=vmax)
    axes[1].set_title("Weight Map on Image", fontsize=11)
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    fname = f"{model_name}_superpixel_weights_{pred_label}.png"
    path = os.path.join(output_dir, fname)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Superpixel chart: {path}")

def run_superpixel_weights_all(models_dict, test_dir=TEST_DIR, output_dir=OUTPUT_DIR, num_samples=1000, top_n=15):
    for model_name, model in models_dict.items():
        for cls_name in CLASS_NAMES:
            cls_dir   = os.path.join(test_dir, cls_name)
            img_files = get_image_files(cls_dir, n=1)
            if not img_files: continue
            try:
                print(f"   {model_name} / {cls_name}")
                plot_superpixel_weights(model, model_name, img_files[0], output_dir, num_samples, top_n)
            except Exception as e:
                print(f"    Failed: {e}")
    print("\n Superpixel weight charts complete.")

# =============================================================
# RUN PIPELINE
# =============================================================
if __name__ == "__main__":
    print("\n" + "="*55)
    print("   TRAINING ALL 3 MODELS — PAPER-READY VERSION")
    print("="*55)
    all_results = {}
    model_vgg16, acc_vgg16 = train_and_evaluate("vgg16")
    model_resnet50, acc_resnet50 = train_and_evaluate("resnet50")
    model_effb0, acc_effb0 = train_and_evaluate("efficientnetb0")
    
    all_results["vgg16"] = acc_vgg16
    all_results["resnet50"] = acc_resnet50
    all_results["efficientnetb0"] = acc_effb0
    
    LOADED_MODELS = {
        "vgg16": model_vgg16,
        "resnet50": model_resnet50,
        "efficientnetb0": model_effb0,
    }

    print("\n" + "="*55)
    print("   FINAL COMPARISON — V2")
    print("="*55)
    for name, acc in all_results.items():
        print(f"{name:<16} {acc*100:.2f}%")
    print("="*55)

    run_gradcam_all(LOADED_MODELS, TEST_DIR, OUTPUT_DIR, samples_per_class=2)
    run_lime_all(LOADED_MODELS, TEST_DIR, OUTPUT_DIR, samples_per_class=2, num_samples=1000, num_features=5)
    run_superpixel_weights_all(LOADED_MODELS, TEST_DIR, OUTPUT_DIR, num_samples=1000, top_n=15)

    print("\n" + "="*55)
    print("   PIPELINE COMPLETE — All files saved:")
    print("="*55)
    output_files = [f for f in sorted(os.listdir(OUTPUT_DIR))
        if any(x in f for x in ["gradcam","lime","superpixel","curves","confusion_matrix"])]
    for f in output_files:
        path    = os.path.join(OUTPUT_DIR, f)
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"{f}  ({size_mb:.2f} MB)")
    print("="*55)

2026-03-17 01:02:17.112768: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773709337.333399      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773709337.400633      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773709337.951152      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773709337.951190      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773709337.951193      24 computation_placer.cc:177] computation placer alr

✅ Config ready
   GPU: True

   TRAINING ALL 3 MODELS — PAPER-READY VERSION

   VGG16
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.

 Class weights (glioma boosted):
   glioma      : 1.3000 (1400 images)
   meningioma  : 1.0000 (1400 images)
   notumor     : 1.0000 (1400 images)
   pituitary   : 1.0000 (1400 images)


I0000 00:00:1773709366.058177      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773709366.064222      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

   Phase 1 — head only | lr=0.0005
Epoch 1/30


I0000 00:00:1773709371.695885      89 service.cc:152] XLA service 0x7f0b1c00efe0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773709371.695943      89 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773709371.695949      89 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1773709372.284088      89 cuda_dnn.cc:529] Loaded cuDNN version 91002


...

   Superpixel chart: /kaggle/working/outputs/efficientnetb0_superpixel_weights_meningioma.png
   efficientnetb0 / notumor


  0%|          | 0/1000 [00:00<?, ?it/s]

   Superpixel chart: /kaggle/working/outputs/efficientnetb0_superpixel_weights_notumor.png
   efficientnetb0 / pituitary


  0%|          | 0/1000 [00:00<?, ?it/s]

   Superpixel chart: /kaggle/working/outputs/efficientnetb0_superpixel_weights_pituitary.png

 Superpixel weight charts complete.

   PIPELINE COMPLETE — All files saved:
efficientnetb0_confusion_matrix.png  (0.07 MB)
efficientnetb0_curves.png  (0.10 MB)
efficientnetb0_gradcam.png  (2.91 MB)
efficientnetb0_lime.png  (2.52 MB)
efficientnetb0_superpixel_weights_glioma.png  (0.13 MB)
efficientnetb0_superpixel_weights_meningioma.png  (0.09 MB)
efficientnetb0_superpixel_weights_notumor.png  (0.11 MB)
efficientnetb0_superpixel_weights_pituitary.png  (0.12 MB)
resnet50_confusion_matrix.png  (0.07 MB)
resnet50_curves.png  (0.12 MB)
resnet50_gradcam.png  (2.46 MB)
resnet50_lime.png  (2.53 MB)
resnet50_superpixel_weights_glioma.png  (0.10 MB)
resnet50_superpixel_weights_meningioma.png  (0.09 MB)
resnet50_superpixel_weights_notumor.png  (0.09 MB)
resnet50_superpixel_weights_pituitary.png  (0.11 MB)
vgg16_confusion_matrix.png  (0.07 MB)
vgg16_curves.png  (0.10 MB)
vgg16_gradcam.png  (2.70 MB)
vgg1

In [2]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [3]:
import os

print("INPUT FOLDERS:")
print(os.listdir("/kaggle/input"))

for folder in os.listdir("/kaggle/input"):
    print("\nInside:", folder)
    print(os.listdir("/kaggle/input/" + folder))

INPUT FOLDERS:
['datasets']

Inside: datasets
['masoudnickparvar', 'avigaur1515']


In [4]:
import os

base = "/kaggle/input/datasets/avigaur1515"

print("Folders inside avigaur1515:")
print(os.listdir(base))

for folder in os.listdir(base):
    path = os.path.join(base, folder)
    if os.path.isdir(path):
        print("\nInside:", folder)
        print(os.listdir(path))

Folders inside avigaur1515:
['phase-3-enet']

Inside: phase-3-enet
['efficientnetb0_p3.keras']


In [5]:
"/kaggle/input/datasets/avigaur1515/phase-3-enet/efficientnetb0_p3.keras"

'/kaggle/input/datasets/avigaur1515/phase-3-enet/efficientnetb0_p3.keras'

In [6]:
# =============================================================
# FULL RESUME CELL — EfficientNet Phase-3 → Evaluation → XAI
# =============================================================

import os, glob, random, warnings, cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_pre
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")

# -------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["glioma","meningioma","notumor","pituitary"]
BATCH_SIZE = 32
OUTPUT_DIR = "/kaggle/working/outputs"
TRAIN_DIR="/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training"
TEST_DIR="/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing"

os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = (240,240)

# -------------------------------------------------------------
# DATA GENERATORS
# -------------------------------------------------------------
def get_generators():

    train_datagen = ImageDataGenerator(
        preprocessing_function=effnet_pre,
        rotation_range=15,
        width_shift_range=0.10,
        height_shift_range=0.10,
        zoom_range=0.10,
        horizontal_flip=True,
        validation_split=0.2
    )

    test_datagen = ImageDataGenerator(preprocessing_function=effnet_pre)

    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        classes=CLASS_NAMES,
        subset="training",
        class_mode="categorical",
        seed=SEED
    )

    val_gen = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        classes=CLASS_NAMES,
        subset="validation",
        class_mode="categorical",
        seed=SEED
    )

    test_gen = test_datagen.flow_from_directory(
        TEST_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        classes=CLASS_NAMES,
        class_mode="categorical",
        shuffle=False
    )

    return train_gen, val_gen, test_gen

# -------------------------------------------------------------
# CLASS WEIGHTS
# -------------------------------------------------------------
def compute_class_weights():

    counts = {}

    for cls in CLASS_NAMES:
        cls_dir = os.path.join(TRAIN_DIR, cls)
        counts[cls] = len(os.listdir(cls_dir))

    total = sum(counts.values())
    weights = {i: total/(len(CLASS_NAMES)*counts[c]) for i,c in enumerate(CLASS_NAMES)}

    return weights

# -------------------------------------------------------------
# CALLBACKS
# -------------------------------------------------------------
def get_callbacks():

    return [
        ModelCheckpoint(
            os.path.join(OUTPUT_DIR,"efficientnet_resume.keras"),
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1
        ),

        EarlyStopping(
            monitor="val_accuracy",
            patience=10,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3
        )
    ]

# -------------------------------------------------------------
# EVALUATION
# -------------------------------------------------------------
def evaluate(model,test_gen):

    test_gen.reset()
    preds=model.predict(test_gen)
    y_pred=np.argmax(preds,axis=1)
    y_true=test_gen.classes

    acc=accuracy_score(y_true,y_pred)
    print("\nAccuracy:",acc*100)

    print("\nClassification Report\n")
    print(classification_report(y_true,y_pred,target_names=CLASS_NAMES))

    cm=confusion_matrix(y_true,y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES)
    plt.title("Confusion Matrix")
    plt.show()

# -------------------------------------------------------------
# RESUME TRAINING
# -------------------------------------------------------------
print("\n===============================")
print("LOADING CHECKPOINT")
print("===============================\n")

checkpoint="/kaggle/input/datasets/avigaur1515/phase-3-enet/efficientnetb0_p3.keras"

model=load_model(checkpoint)

print("Checkpoint loaded successfully")

train_gen,val_gen,test_gen=get_generators()

class_weights=compute_class_weights()

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n===============================")
print("RESUMING TRAINING (Epoch 9→35)")
print("===============================\n")

history=model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=35,
    initial_epoch=9,
    class_weight=class_weights,
    callbacks=get_callbacks()
)

# -------------------------------------------------------------
# SAVE FINAL MODEL
# -------------------------------------------------------------
final_path=os.path.join(OUTPUT_DIR,"efficientnet_final.keras")
model.save(final_path)

print("\nFinal model saved:",final_path)

# -------------------------------------------------------------
# EVALUATE
# -------------------------------------------------------------
evaluate(model,test_gen)

print("\nPipeline finished. Outputs in:",OUTPUT_DIR)


LOADING CHECKPOINT

Checkpoint loaded successfully
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.

RESUMING TRAINING (Epoch 9→35)

Epoch 10/35
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 400ms/step - accuracy: 0.9971 - loss: 0.0514
Epoch 10: val_accuracy improved from -inf to 0.97589, saving model to /kaggle/working/outputs/efficientnet_resume.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 111s 556ms/step - accuracy: 0.9971 - loss: 0.0514 - val_accuracy: 0.9759 - val_loss: 0.1362 - learning_rate: 1.0000e-05
Epoch 11/35
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 404ms/step - accuracy: 0.9975 - loss: 0.0526
Epoch 11: val_accuracy improved from 0.97589 to 0.97857, saving model to /kaggle/working/outputs/efficientnet_resume.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 509ms/step - accuracy: 0.9975 - loss: 0.0526 - val_accuracy: 0.9786 - val_loss: 0.1253 - learning_rate: 1.0000e-05
Epoch 12/35
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/step - accuracy: 0.99